In [ ]:
import torch
from torch import nn
from tqdm import tqdm

In [ ]:
TRAIN_SPLIT = 0.9
BLOCK_SIZE = 8
BATCH_SIZE = 32
LEARNING_RATE = 1e-2
MAX_ITERS = 3000
EVAL_ITERS = 200

RANDOM_SEED = 1337
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
with open("input.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("Length of the text:", len(text))

chars = sorted(list(set(text)))
print("Unique characters:", chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [char_to_idx[c] for c in s]
decode = lambda l: ''.join([idx_to_char[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print("Data tensor shape:", data.shape)
print("First 100 characters of the data tensor:", data[:100])

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"Context: {context.tolist()} -> Target: {target.item()}")

In [ ]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i + 1:i + BLOCK_SIZE + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print("Inputs (x):", xb.shape, xb)
print("Targets (y):", yb.shape, yb)

for b in range(BATCH_SIZE):
    for t in range(BLOCK_SIZE):
        context = xb[b, :t + 1]
        target = yb[b, t]
        print(f"Batch {b}, Time {t}: Context: {context.tolist()} -> Target: {target.item()}")

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        logits = logits.view(B * T, C)
        targets = targets.view(B * T)

        loss = nn.functional.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus on last time step
            logits = logits[:, -1, :]
            probabilities = nn.functional.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probabilities, num_samples=1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

torch.manual_seed(RANDOM_SEED)
model = BigramLanguageModel(vocab_size=len(chars))
model = model.to(device)
logits, loss = model(xb, yb)

print(loss)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_sequence = model.generate(context, max_new_tokens=100)
print("Generated sequence:", decode(generated_sequence[0].tolist()))

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

pbar = tqdm(range(MAX_ITERS))
for step in pbar:
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % EVAL_ITERS == 0:
        losses = estimate_loss()
        print(f"Step {step}: Train Loss: {losses['train']:.4f}, Val Loss: {losses['val']:.4f}")
    pbar.set_description(f"Step {step}, Loss: {loss.item():.4f}")

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_sequence = model.generate(context, max_new_tokens=300)
print("Generated sequence:", decode(generated_sequence[0].tolist()))